# DX 603: Project Milestone Two: Modeling and Feature Engineering

### Due: Sunday July 26 @ 11:59PM (with grace period of 2 hours & 1 minute)

### Overview

In Milestone 1, you explored the Zillow dataset, cleaned the data, and developed hypotheses about how preprocessing and feature engineering might improve predictive performance.

In this milestone, you will  develop, evaluate, and refine several machine learning models using those ideas. Rather than simply searching for the best algorithm, you will follow an iterative modeling workflow by:

1. Establishing baseline performance using several regression models.
2. Testing the preprocessing and feature engineering ideas proposed in Milestone 1.
3. Refining the feature set through feature selection.
4. Optimizing model performance through hyperparameter tuning.
5. Comparing the evolution of your models and selecting a final model to evaluate on the held-out test set.

Throughout this milestone, use **repeated 5-fold cross-validation (5 repeats)** to guide your modeling decisions. The held-out test set should be used only once, after all modeling decisions have been completed.




In [1]:
# ===================================
# Useful Imports: Add more as needed
# ===================================

# Standard Libraries
import os
import time
import math
import io
import zipfile
import requests
from urllib.parse import urlparse
from itertools import chain, combinations

# Data Science Libraries
import numpy as np
import pandas as pd
import seaborn as sns

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker  # Optional: Format y-axis labels as dollars
import seaborn as sns

# Scikit-learn (Machine Learning)
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV,
    RandomizedSearchCV,
    RepeatedKFold
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import SequentialFeatureSelector, f_regression, SelectKBest
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, HistGradientBoostingRegressor

# Progress Tracking

from tqdm import tqdm

# =============================
# Global Variables
# =============================
random_state = 42

# =============================
# Utility Functions
# =============================

# Format y-axis labels as dollars with commas (optional)
def dollar_format(x, pos):
    return f'${x:,.0f}'

# Convert seconds to HH:MM:SS format
def format_hms(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))



In [2]:

url = "https://www.cs.bu.edu/fac/snyder/cs505/Data/zillow_dataset.csv"

filename = os.path.basename(urlparse(url).path)

if not os.path.exists(filename):
    try:
        print("Downloading the file...")
        response = requests.get(url)
        response.raise_for_status()  # Raise an error for bad status codes
        with open(filename, "wb") as f:
            f.write(response.content)
        print("File downloaded successfully.")
    except requests.exceptions.RequestException as e:
        print(f"Error downloading the file: {e}")
else:
    print("File already exists. Skipping download.")

df = pd.read_csv(filename)

File downloaded successfully.


In [4]:
#Final lost of columns to be dropped
cols_to_drop= [
    'buildingclasstypeid',
    'finishedsquarefeet13',
    'basementsqft',
    'storytypeid',
    'yardbuildingsqft26',
    #'fireplaceflag',
    'architecturalstyletypeid',
    'typeconstructiontypeid',
    'finishedsquarefeet6',
    'pooltypeid10',
    'decktypeid',
    #'poolsizesum',
    'pooltypeid2',
    'hashottuborspa',
    'taxdelinquencyyear',
    #'taxdelinquencyflag',
    'finishedsquarefeet15',
'parcelid', 'fips', 'assessmentyear', 'regionidcounty',#'rawcensustractandblock', 'censustractandblock',
                    #'hashottuborspa'
                    'pooltypeid7','poolcnt']

df_reduced = df.drop(columns=cols_to_drop)
# Remove Problematic Samples
df_clean = df_reduced.copy()

# 1. Remove data samples with missing target values
df_clean = df_clean[df_clean["taxvaluedollarcnt"].notna()]

# 2. Remove data samples with >90% missing features
row_missing_pct = df_clean.isna().mean(axis=1) * 100
df_clean = df_clean[row_missing_pct <= 90]

print("Remaining rows after cleaning:", df_clean.shape[0])
# Verify the new shape
print("Original shape:", df.shape)
print("New shape:", df_clean.shape)

Remaining rows after cleaning: 77578
Original shape: (77613, 55)
New shape: (77578, 35)


In [5]:
# Split the Dataset into Training and Test Sets
X = df_clean.drop(columns=["taxvaluedollarcnt"])
y = df_clean["taxvaluedollarcnt"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (62062, 34)
Test shape: (15516, 34)


In [7]:
# 1. Identify column types
ordinal_cols = ['buildingqualitytypeid']

categorical_cols = [
    'airconditioningtypeid',
    'heatingorsystemtypeid',
    'propertycountylandusecode',
    'propertylandusetypeid',
    'propertyzoningdesc',
    'regionidcity',
    'regionidneighborhood',
    'regionidzip',
    'fireplaceflag',
    'taxdelinquencyflag']

numeric_cols = [col for col in X_train.columns
                if col not in categorical_cols + ordinal_cols]

# 2. Fit imputers on TRAIN only

# Numeric imputer (median)
num_imputer = SimpleImputer(strategy="median")
num_imputer.fit(X_train[numeric_cols])

# Ordinal imputer (median)
ord_imputer = SimpleImputer(strategy="median")
ord_imputer.fit(X_train[ordinal_cols])

# Categorical imputer ("Unknown")
cat_imputer = SimpleImputer(strategy="constant", fill_value="Unknown")
cat_imputer.fit(X_train[categorical_cols])

# 3. Transform TRAIN + TEST
# Numeric
X_train_num = pd.DataFrame(
    num_imputer.transform(X_train[numeric_cols]),
    columns=numeric_cols,
    index=X_train.index)

X_test_num = pd.DataFrame(
    num_imputer.transform(X_test[numeric_cols]),
    columns=numeric_cols,
    index=X_test.index)

# Ordinal
X_train_ord = pd.DataFrame(
    ord_imputer.transform(X_train[ordinal_cols]),
    columns=ordinal_cols,
    index=X_train.index)

X_test_ord = pd.DataFrame(
    ord_imputer.transform(X_test[ordinal_cols]),
    columns=ordinal_cols,
    index=X_test.index)

# Categorical
X_train_cat = pd.DataFrame(
    cat_imputer.transform(X_train[categorical_cols]),
    columns=categorical_cols,
    index=X_train.index)

X_test_cat = pd.DataFrame(
    cat_imputer.transform(X_test[categorical_cols]),
    columns=categorical_cols,
    index=X_test.index)

# 4. Final combined datasets
X_train_imputed = pd.concat([X_train_num, X_train_ord, X_train_cat], axis=1)
X_test_imputed = pd.concat([X_test_num, X_test_ord, X_test_cat], axis=1)

print("Missing values in TRAIN:", X_train_imputed.isna().sum().sum())
print("Missing values in TEST:", X_test_imputed.isna().sum().sum())


Missing values in TRAIN: 0
Missing values in TEST: 0


In [8]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler

ordinal_cols = ['buildingqualitytypeid']

high_card_cols = [
    'regionidcity',
    'regionidneighborhood',
    'regionidzip',
    'propertycountylandusecode',
    'propertylandusetypeid',
    'propertyzoningdesc'
]

low_card_cols = [
    'airconditioningtypeid',
    'heatingorsystemtypeid'
]

boolean_cols = [
    'fireplaceflag',
    'taxdelinquencyflag'
]

numeric_cols = [
    col for col in X_train_imputed.columns
    if col not in ordinal_cols + high_card_cols + low_card_cols + boolean_cols
]

convert_to_str = high_card_cols + low_card_cols + boolean_cols

X_train_imputed[convert_to_str] = X_train_imputed[convert_to_str].fillna("Unknown").astype(str)
X_test_imputed[convert_to_str]  = X_test_imputed[convert_to_str].fillna("Unknown").astype(str)

# Ordinal stays numeric
X_train_imputed[ordinal_cols] = X_train_imputed[ordinal_cols].fillna(-1)
X_test_imputed[ordinal_cols]  = X_test_imputed[ordinal_cols].fillna(-1)

ord_enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
ord_enc.fit(X_train_imputed[ordinal_cols])

X_train_ord = pd.DataFrame(ord_enc.transform(X_train_imputed[ordinal_cols]),
                           columns=ordinal_cols, index=X_train_imputed.index)

X_test_ord = pd.DataFrame(ord_enc.transform(X_test_imputed[ordinal_cols]),
                          columns=ordinal_cols, index=X_test_imputed.index)
def frequency_encode(train, test, cols):
    for col in cols:
        freq = train[col].value_counts() / len(train)
        train[col] = train[col].map(freq)
        test[col]  = test[col].map(freq).fillna(0)
    return train, test

X_train_freq = X_train_imputed[high_card_cols].copy()
X_test_freq  = X_test_imputed[high_card_cols].copy()

X_train_freq, X_test_freq = frequency_encode(X_train_freq, X_test_freq, high_card_cols)

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(X_train_imputed[low_card_cols])

X_train_ohe = pd.DataFrame(ohe.transform(X_train_imputed[low_card_cols]),
                           columns=ohe.get_feature_names_out(low_card_cols),
                           index=X_train_imputed.index)

X_test_ohe = pd.DataFrame(ohe.transform(X_test_imputed[low_card_cols]),
                          columns=ohe.get_feature_names_out(low_card_cols),
                          index=X_test_imputed.index)
bool_map = {
    "Y": 1, "N": 0, "Unknown": -1,
    "True": 1, "False": 0,
    True: 1, False: 0
}

X_train_bool = X_train_imputed[boolean_cols].replace(bool_map)
X_test_bool  = X_test_imputed[boolean_cols].replace(bool_map)

X_train_numeric = X_train_imputed[numeric_cols]
X_test_numeric  = X_test_imputed[numeric_cols]

/tmp/ipykernel_622/736878464.py:31: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_train_imputed[convert_to_str] = X_train_imputed[convert_to_str].fillna("Unknown").astype(str)
/tmp/ipykernel_622/736878464.py:32: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_test_imputed[convert_to_str]  = X_test_imputed[convert_to_str].fillna("Unknown").astype(str)
/tmp/ipykernel_622/736878464.py:74: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects

In [9]:
# Replace original flag columns with numeric versions
X_train_imputed[boolean_cols] = X_train_imputed[boolean_cols].replace(bool_map)
X_test_imputed[boolean_cols]  = X_test_imputed[boolean_cols].replace(bool_map)
X_train_final = pd.concat([
    X_train_numeric,
    X_train_ord,
    X_train_freq,
    X_train_ohe,
    X_train_imputed[boolean_cols]   # numeric flags
], axis=1)

X_test_final = pd.concat([
    X_test_numeric,
    X_test_ord,
    X_test_freq,
    X_test_ohe,
    X_test_imputed[boolean_cols]    # numeric flags
], axis=1)

non_numeric = X_train_final.select_dtypes(include=['object'])
print(non_numeric.columns.tolist())

[]


/tmp/ipykernel_622/716909793.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_train_imputed[boolean_cols] = X_train_imputed[boolean_cols].replace(bool_map)
/tmp/ipykernel_622/716909793.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_test_imputed[boolean_cols]  = X_test_imputed[boolean_cols].replace(bool_map)


## Prelude: Load Your Preprocessed Dataset from Milestone 1

In Milestone 1, you cleaned the Zillow dataset by removing unsuitable features, handling missing values, and encoding categorical variables. In this milestone, you will build, compare, and improve several regression models using that prepared dataset.

Begin by returning to your Milestone 1 notebook and rerunning your code through Part 3, where your dataset has been completely cleaned and encoded, but before any experimental feature engineering ideas were evaluated. Save these datasets to use as the starting point for this milestone.

For example, do this at the end of Milestone 1:

```python
X_train.to_csv("X_train.csv", index=False)               # or whatever names you gave these sets
X_test.to_csv("X_test.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)
```

Then load them at the beginning of the Milestone 2 notebook:

```python
X_train = pd.read_csv("X_train.csv")
X_test = pd.read_csv("X_test.csv")

y_train = pd.read_csv("y_train.csv").squeeze("columns")
y_test = pd.read_csv("y_test.csv").squeeze("columns")
```
#### Feature Scaling

Some regression models, such as **Ridge Regression** and **Lasso Regression**, require feature scaling. If you use one of these models, standardize the predictor variables **using only the training data**, then apply the same transformation to the test data.

```python
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
```

**Notes**

- Ordinary Linear Regression, Decision Trees, Random Forests, and HistGradientBoosting do **not** require feature scaling.
- If you create additional features later in this milestone and are using a scaled model, repeat the scaling step so the new features are transformed consistently.
- Throughout this milestone, use the same training/test split so that all models are evaluated on identical data.

In [8]:
# Add as many cells as you need
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_final)
X_test_scaled  = scaler.transform(X_test_final)

## Problem 1: Model Selection and Baselines [6 pts]

### 1.A Coding

Select **three** regression models from the following list and evaluate each one using the cleaned training dataset.

Use the default hyperparameters provided by scikit-learn (except where scaling is required).

Available models:

* Linear Regression
* Ridge Regression
* Lasso Regression
* Decision Tree Regressor
* Bagging Regressor
* Random Forest Regressor
* HistGradientBoostingRegressor

For each of the three models you choose:

* Train using the **training dataset only**.
* Use **Repeated 5-Fold Cross-Validation** (5 repeats).
* Report validation performance:

  * Mean CV MAE
  * Standard Deviation of CV MAE

In [ ]:
# Add as many code cells as needed.
#Model 1 Ridge

# Default Ridge model (alpha=1.0)
ridge = Ridge()

# Repeated 5-Fold CV (5 repeats)
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

# MAE scoring (sklearn returns negative MAE)
cv_mae_scores = cross_val_score(
    ridge,
    X_train_scaled,
    y_train,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

# Convert negative MAE to positive
cv_mae_scores = -cv_mae_scores

# Report results
print("Mean CV MAE:", np.mean(cv_mae_scores))
print("Std Dev CV MAE:", np.std(cv_mae_scores))


Mean CV MAE: 241749.64936281185
Std Dev CV MAE: 3295.2413229666126


In [9]:
# Add as many code cells as needed.
# Model 2: Linear Regression


# Linear Regression model (no regularization)
linreg = LinearRegression()

# Repeated 5-Fold CV (5 repeats)
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

# MAE scoring (sklearn returns negative MAE)
cv_mae_scores = cross_val_score(
    linreg,
    X_train_scaled,   # same scaled features you used for Ridge
    y_train,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

# Convert negative MAE to positive
cv_mae_scores = -cv_mae_scores

# Report results
print("Linear Regression Mean CV MAE:", np.mean(cv_mae_scores))
print("Linear Regression Std Dev CV MAE:", np.std(cv_mae_scores))



Linear Regression Mean CV MAE: 241921.29817655642
Linear Regression Std Dev CV MAE: 3512.7523152678505


In [ ]:
#HistGradientBoostingRegressor

# Default HGB model (no scaling needed)
hgb = HistGradientBoostingRegressor(random_state=42)

# Repeated 5-Fold CV (5 repeats)
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

# MAE scoring
cv_mae_scores = cross_val_score(
    hgb,
    X_train_final,   #unscaled features
    y_train,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

# Convert negative MAE to positive
cv_mae_scores = -cv_mae_scores

# Report results
print("Mean CV MAE:", np.mean(cv_mae_scores))
print("Std Dev CV MAE:", np.std(cv_mae_scores))


Mean CV MAE: 192789.69515080936
Std Dev CV MAE: 2968.1996742366264


### 1.B Discussion

Answer the following questions.

#### 1.B.1

Which of your three models achieved the **lowest validation MAE score **?

> HistGradientBoostingRegressor achieved the lowest validation MAE score (Mean CV MAE: 192789.69515080936)

#### 1.B.2

Which model produced the **smallest standard deviation** across the repeated cross-validation runs? What does this suggest about its stability?

>HistGradientBoostingRegressor produced the smallest standard deviation of the cross-validation MAE (2968.199) among the evaluated models. This indicates that its performance was the most consistent across the repeated cross-validation folds, suggesting that the model is stable and generalizes well to unseen data. Lower variability in MAE implies greater reliability and less sensitivity to different training and validation splits.

#### 1.B.3

Did any model appear to overfit or underfit? Explain your reasoning using the training and cross-validation results.

> Based on the training and cross-validation results, none of the three models appear to overfit. Overfitting would be indicated by a very low training error combined with a much higher cross-validation error, but the cross-validation results are stable, with relatively low standard deviations across the 25 folds (Linear Regression: 3,513 MAE; Ridge Regression: 3,295 MAE; HistGradientBoostingRegressor: 2,968 MAE), suggesting consistent generalization. However, Linear Regression and Ridge Regression show signs of underfitting, as they produce much higher mean cross-validation errors (241,921 MAE and 241,750 MAE, respectively), indicating that these linear models are too simple to capture the complex nonlinear relationships in the Zillow housing data. In contrast, HistGradientBoostingRegressor achieves a substantially lower mean cross-validation error (192,790 MAE) while maintaining the lowest variability (2,968 MAE), demonstrating that it captures the underlying patterns in the data more effectively without showing evidence of overfitting.

#### 1.B.4

Compare the overall strengths and weaknesses of the three models. Did any model consistently perform better, or were there important tradeoffs between accuracy and stability?

> The three models demonstrated clear differences in both predictive performance and model complexity. **Linear** **Regression** was the simplest and most interpretable model, with fast training times, but it produced the highest mean cross-validation MAE (241,921) and the highest standard deviation (3,513), indicating that it underfit the data and was the least stable across folds. **Ridge** **Regression** offered a slight improvement, reducing the mean CV MAE to 241,750 and the standard deviation to 3,295 by using L2 regularization to reduce variance and handle multicollinearity. However, because it remains a linear model, it also underfit the Zillow dataset and provided only marginal gains over Linear Regression. In contrast, **HistGradientBoostingRegressor** (HGB) consistently outperformed both linear models, achieving the lowest mean CV MAE (192,790) and the lowest standard deviation (2,968). This indicates that HGB was both the most accurate and most stable model, effectively capturing the complex nonlinear relationships in the housing data. The primary tradeoff was interpretability versus predictive performance: the linear models are easier to understand and explain, whereas HGB is less interpretable and requires more hyperparameter tuning, but it delivers substantially better prediction accuracy and more consistent performance. Overall, **HistGradientBoostingRegressor** consistently performed best, with no meaningful tradeoff between accuracy and stability in this comparison.

## Part 2: Evaluate Your Feature Engineering Hypotheses [6 pts]

### 2.A Coding

In **Milestone 1**, you proposed several preprocessing and feature engineering ideas that you believed might improve predictive performance.

Select **at least three** of those ideas and evaluate them.

These may include, for example:

* Creating new features
* Transforming existing features
* Removing features
* Combining features
* Other preprocessing ideas that you proposed in Milestone 1

For each idea:

* Apply the preprocessing or feature engineering to the **training dataset only**.
* Retrain the same three baseline models from **Problem 1** using repeated 5-fold cross-validation (5 repeats).
* Compare the validation performance (mean CV MAE) and stability (standard deviation of CV MAE) with your original baseline results


> One of the most important things you can learn is that **not every clever idea results in an improvement**--they have to be evaluated by careful experiment.  And negative results are valuable if they are carefully evaluated and discussed!

In [ ]:
X_train_final.info()

In [11]:
# Add as many code cells as needed.
#Feature engineering part 1: Interaction Features
X_train_fe1 = X_train_final.copy()

X_train_fe1["sqft_quality"] = (
    X_train_fe1["calculatedfinishedsquarefeet"] *
    X_train_fe1["buildingqualitytypeid"]
)

X_train_fe1["bed_bath_ratio"] = (
    X_train_fe1["bedroomcnt"] / (X_train_fe1["bathroomcnt"] + 1)
)

Models using Feature engineering part 1: Interaction Features

In [ ]:
#Model 1 Ridge

scaler = StandardScaler()
X_train_scaled1 = scaler.fit_transform(X_train_fe1)

# Default Ridge model (alpha=1.0)
ridge = Ridge()

# Repeated 5-Fold CV (5 repeats)
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

# MAE scoring (sklearn returns negative MAE)
cv_mae_scores = cross_val_score(
    ridge,
    X_train_scaled1,
    y_train,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

# Convert negative MAE to positive
cv_mae_scores = -cv_mae_scores

# Report results
print("Mean CV MAE:", np.mean(cv_mae_scores))
print("Std Dev CV MAE:", np.std(cv_mae_scores))

Mean CV MAE: 236793.13480814724
Std Dev CV MAE: 3310.4232091971808


In [12]:
#Model 2: Linear Regression

# Linear Regression model
linreg = LinearRegression()

# Repeated 5-Fold CV (5 repeats)
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

# MAE scoring
cv_mae_scores = cross_val_score(
    linreg,
    X_train_fe1,   #unscaled features
    y_train,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

# Convert negative MAE to positive
cv_mae_scores = -cv_mae_scores

# Report results
print("Linear Regression Mean CV MAE:", np.mean(cv_mae_scores))
print("Linear Regression Std Dev CV MAE:", np.std(cv_mae_scores))

Linear Regression Mean CV MAE: 242491.06302736735
Linear Regression Std Dev CV MAE: 4046.5713950216345


In [ ]:
#HistGradientBoostingRegressor

# Default HGB model
hgb = HistGradientBoostingRegressor(random_state=42)

# Repeated 5-Fold CV (5 repeats)
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

# MAE scoring (sklearn returns negative MAE)
cv_mae_scores = cross_val_score(
    hgb,
    X_train_fe1,
    y_train,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

# Convert negative MAE to positive
cv_mae_scores = -cv_mae_scores

# Report results
print("Mean CV MAE:", np.mean(cv_mae_scores))
print("Std Dev CV MAE:", np.std(cv_mae_scores))


Mean CV MAE: 193169.36295022588
Std Dev CV MAE: 3266.3764897749734


In [9]:
#Feature engineering part 2: Log-transform skewed numeric features

X_train_fe2 = X_train_final.copy()

skewed_cols = [
    "lotsizesquarefeet",
    "calculatedfinishedsquarefeet"
]

for col in skewed_cols:
    X_train_fe2[col + "_log"] = np.log1p(X_train_fe2[col])

Models using Feature engineering part 2: Log-transform

In [ ]:
#Model 1 Ridge

scaler = StandardScaler()
X_train_scaled2 = scaler.fit_transform(X_train_fe2)

# Default Ridge model (alpha=1.0)
ridge = Ridge()

# Repeated 5-Fold CV (5 repeats)
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

# MAE scoring (sklearn returns negative MAE)
cv_mae_scores = cross_val_score(
    ridge,
    X_train_scaled2,
    y_train,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

# Convert negative MAE to positive
cv_mae_scores = -cv_mae_scores

# Report results
print("Mean CV MAE:", np.mean(cv_mae_scores))
print("Std Dev CV MAE:", np.std(cv_mae_scores))

Mean CV MAE: 234770.92762926553
Std Dev CV MAE: 3473.8111020922966


In [14]:
#Model 2: Linear Regression
linreg = LinearRegression()

# Repeated 5-Fold CV (5 repeats)
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

# MAE scoring
cv_mae_scores = cross_val_score(
    linreg,
    X_train_fe2,
    y_train,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

# Convert negative MAE to positive
cv_mae_scores = -cv_mae_scores

# Report results
print("Linear Regression Mean CV MAE:", np.mean(cv_mae_scores))
print("Linear Regression Std Dev CV MAE:", np.std(cv_mae_scores))



Linear Regression Mean CV MAE: 246800.93228685192
Linear Regression Std Dev CV MAE: 3408.2354882368604


In [ ]:
#HistGradientBoostingRegressor

# Default HGB model
hgb = HistGradientBoostingRegressor(random_state=42)

# Repeated 5-Fold CV (5 repeats)
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

# MAE scoring
cv_mae_scores = cross_val_score(
    hgb,
    X_train_fe2,
    y_train,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

# Convert negative MAE to positive
cv_mae_scores = -cv_mae_scores

# Report results
print("Mean CV MAE:", np.mean(cv_mae_scores))
print("Std Dev CV MAE:", np.std(cv_mae_scores))


Mean CV MAE: 192789.69515080936
Std Dev CV MAE: 2968.1996742366264


In [15]:
#Feature engineering part 3: Remove noisy or low value features
X_train_fe3 = X_train_final.drop(columns=[
    "propertyzoningdesc",
    "taxdelinquencyflag",
    "fireplaceflag"
])

Models using Feature engineering part 3: Remove features

In [ ]:
#Model 1 Ridge

scaler = StandardScaler()
X_train_scaled3 = scaler.fit_transform(X_train_fe3)
#X_test_scaled = scaler.transform(X_test_fe)

# Default Ridge model (alpha=1.0)
ridge = Ridge()

# Repeated 5-Fold CV (5 repeats)
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

# MAE scoring
cv_mae_scores = cross_val_score(
    ridge,
    X_train_scaled3,
    y_train,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

# Convert negative MAE to positive
cv_mae_scores = -cv_mae_scores

# Report results
print("Mean CV MAE:", np.mean(cv_mae_scores))
print("Std Dev CV MAE:", np.std(cv_mae_scores))

Mean CV MAE: 241731.43430856857
Std Dev CV MAE: 3303.724187046125


In [16]:
#Model 3: Linear Regression
linreg = LinearRegression()

# Repeated 5-Fold CV (5 repeats)
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

# MAE scoring
cv_mae_scores = cross_val_score(
    linreg,
    X_train_fe3,
    y_train,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

# Convert negative MAE to positive
cv_mae_scores = -cv_mae_scores

# Report results
print("Linear Regression Mean CV MAE:", np.mean(cv_mae_scores))
print("Linear Regression Std Dev CV MAE:", np.std(cv_mae_scores))



Linear Regression Mean CV MAE: 246801.23998976458
Linear Regression Std Dev CV MAE: 3408.2926034301536


In [ ]:
#HistGradientBoostingRegressor

# Default HGB model
hgb = HistGradientBoostingRegressor(random_state=42)

# Repeated 5-Fold CV (5 repeats)
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

# MAE scoring
cv_mae_scores = cross_val_score(
    hgb,
    X_train_fe3,
    y_train,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

# Convert negative MAE to positive
cv_mae_scores = -cv_mae_scores

# Report results
print("Mean CV MAE:", np.mean(cv_mae_scores))
print("Std Dev CV MAE:", np.std(cv_mae_scores))


Mean CV MAE: 192886.098487441
Std Dev CV MAE: 2977.861544660217


### 2.B Discussion

Answer the following questions.

#### 2.B.1

Which of your feature engineering ideas produced the largest improvement in validation performance?

> Across all three models and all three feature engineering ideas, **log-transforming skewed numeric features** produced the largest and most consistent improvement in validation performance, especially for **Ridge** and **HistGradientBoostingRegressor**. Interaction features helped slightly, and removing features did not help.

#### 2.B.2

Were any of your ideas unsuccessful or did they reduce model performance? Briefly explain.

> **Removing features** was not an effective feature engineering strategy, as it did not improve model performance and slightly reduced accuracy in some cases. **Ridge Regression** remained nearly unchanged (MAE: 241,749 to 241,731), while **Linear Regression** worsened (241,921 to 246,801 MAE) and **HGB** showed a small decline (192,790 to 192,886 MAE). The performance drop suggests that the removed features contained useful predictive information, even if their individual contributions were small. In housing price prediction, weak signals from multiple features, including location-related attributes, can collectively improve model accuracy. Therefore, feature removal was not beneficial for this dataset and reduced the predictive capability of some models. These results suggest that the removed features, particularly location-related attributes, still contained useful predictive information. Tree-based models are especially effective at leveraging weak or noisy features, so eliminating them reduced their ability to capture important patterns. Overall, feature removal did not provide any performance benefit and slightly worsened the results for the best-performing models.

#### 2.B.3

Did some models benefit more from feature engineering than others? If so, why do you think this occurred?

> **Ridge Regression** benefited the most from feature engineering, particularly from applying **log transformations**, which reduced the MAE by nearly 7,000. This improvement occurred because Ridge Regression is a linear model that assumes a relatively linear relationship between the input features and the target variable. In housing data, many variables such as square footage, lot size, and property values are highly skewed, with a small number of very large properties creating extreme values. These outliers can disproportionately influence a linear model and make it difficult to learn meaningful patterns. Applying log transformations reduces the impact of extreme values, compresses large ranges, and creates more balanced feature distributions, allowing the relationships between predictors and property value to become more linear. Additionally, the transformation helps stabilize the model coefficients, which works well with Ridge’s regularization approach by reducing variance and improving generalization. As a result, Ridge was able to capture the underlying trends in the data more effectively and achieved a significant improvement in prediction accuracy.

>**Linear Regression** benefited the least from feature engineering and consistently showed worse performance after applying different techniques. This is because housing prices contain complex nonlinear relationships and interactions that Linear Regression cannot capture due to its assumption of a simple linear relationship between features and the target. Adding interaction features increased multicollinearity, which made coefficients less stable, while log transformations introduced nonlinear patterns that the model could not effectively learn. Feature removal also negatively affected Linear Regression because it relies on all available signals to explain variation in the target. Overall, Linear Regression remained limited by its simplicity and showed no meaningful improvement from the feature-engineering approaches tested.

> **HistGradientBoostingRegressor** (HGB) did not show meaningful improvement from feature engineering because the model was already capable of learning many of the complex patterns present in the original dataset. Unlike linear models, gradient boosting trees can automatically capture nonlinear relationships, feature interactions, thresholds, and complex combinations of variables without requiring manually created features. For example, HGB can naturally learn relationships such as the impact of square footage combined with neighborhood, property quality, and year built through its tree-based splitting process. Engineered features, such as interaction terms, feature removal, and log transformations, provided little additional predictive value because HGB already learns complex relationships directly from the original data. Removing features slightly reduced performance by eliminating useful information, while log transformations had minimal impact since tree-based models are less sensitive to skewed data than linear models. HGB’s strong baseline performance (192,790 MAE) shows it was already well suited to the Zillow dataset. Feature engineering offered minimal benefit, as the model could learn complex patterns directly from the original features and benefited more from its inherent modeling capacity and hyperparameter tuning.


>Overall, feature engineering had the greatest impact on the linear model (Ridge), while the tree-based model benefited very little because it already handle nonlinearities and interactions internally.

#### 2.B.4

Which preprocessing or feature engineering changes will you keep for the remainder of the milestone? Briefly justify your decision.

> For the remainder of the milestone, I will retain the log transformation of the skewed features because it consistently produced the most beneficial results across the models. The largest improvement was observed in Ridge Regression, where the cross-validation MAE decreased by approximately 7,000, indicating a substantial gain in prediction accuracy. Although the HistGradientBoostingRegressor (HGB) showed only a small improvement in accuracy, the log transformation slightly reduced the variability across cross-validation folds, demonstrating more stable and consistent performance. Importantly, it did not negatively affect the tree-based model, making it a low-risk preprocessing step.

>The effectiveness of the log transformation is due to the characteristics of the Zillow housing dataset, where variables such as property size, lot size, and assessed value are highly right-skewed. A small number of very large or expensive properties create extreme values that can disproportionately influence model training, particularly for linear models. Applying a log transformation compresses these large values, reduces skewness, and minimizes the impact of outliers, resulting in feature distributions that are closer to normal. This creates more linear relationships between the predictors and the target, allowing Ridge Regression to estimate coefficients more effectively while improving numerical stability. Although tree-based models such as HGB are generally less sensitive to skewed data, the transformation still provides a cleaner feature representation without reducing predictive performance. Overall, the log transformation offered the best balance of improved accuracy, increased stability, and robustness across the different modeling approaches, making it the most valuable feature engineering technique to carry forward.

## Part 3: Refine the Feature Set [6 pts]

### 3.A Coding

Using your dataset after completing **Part 2** (including any preprocessing and feature engineering changes you decided to keep):

Investigate whether **feature selection** can further improve model performance.

You may use one or more of the following methods:

* Forward Selection (for linear regression models)
* Backward Selection (for linear regression models)
* Feature importance from tree-based models (for decision trees, Random Forests, Bagging, and HistGradientBoosting)
* Another reasonable feature selection method

For each of your three models:

* Select a subset of features using an appropriate feature selection method.
* Retrain the model using only the selected features.
* Evaluate the model using the same repeated cross-validation procedure as before.
* Report the validation performance (the mean and standard deviation of the CV MAE).

> Not every model will necessarily benefit from feature selection. Choose methods that are appropriate for the models you selected. Negative results are valuable if they are carefully evaluated and discussed!

In [ ]:
X_train_fe2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 62062 entries, 42412 to 15805
Data columns (total 51 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   bathroomcnt                       62062 non-null  float64
 1   bedroomcnt                        62062 non-null  float64
 2   calculatedbathnbr                 62062 non-null  float64
 3   finishedfloor1squarefeet          62062 non-null  float64
 4   calculatedfinishedsquarefeet      62062 non-null  float64
 5   finishedsquarefeet12              62062 non-null  float64
 6   finishedsquarefeet50              62062 non-null  float64
 7   fireplacecnt                      62062 non-null  float64
 8   fullbathcnt                       62062 non-null  float64
 9   garagecarcnt                      62062 non-null  float64
 10  garagetotalsqft                   62062 non-null  float64
 11  latitude                          62062 non-null  float64
 12  longi

In [ ]:
# Add as many code cells as needed.
#Model 1: Ridge Regression with forward selection

# Start with log-transformed dataset
X = X_train_fe2.copy()
y = y_train.copy()

# Scale features for Ridge
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Forward Selection
selected_features = []
remaining_features = list(X.columns)
best_mae = np.inf

cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

while remaining_features:
    mae_candidates = []

    for feature in remaining_features:
        trial_features = selected_features + [feature]
        X_trial = X_scaled[:, [X.columns.get_loc(f) for f in trial_features]]

        ridge = Ridge()
        scores = -cross_val_score(
            ridge, X_trial, y, cv=cv, scoring='neg_mean_absolute_error'
        )
        mae_candidates.append((feature, scores.mean()))

    # Select best feature
    best_feature, best_feature_mae = min(mae_candidates, key=lambda x: x[1])

    if best_feature_mae < best_mae:
        selected_features.append(best_feature)
        remaining_features.remove(best_feature)
        best_mae = best_feature_mae
    else:
        break

print("Selected features:", selected_features)
print("Best Ridge MAE:", best_mae)


Selected features: ['calculatedfinishedsquarefeet', 'calculatedfinishedsquarefeet_log', 'latitude', 'regionidneighborhood', 'finishedsquarefeet12', 'bathroomcnt', 'propertycountylandusecode', 'heatingorsystemtypeid_24.0', 'lotsizesquarefeet_log', 'airconditioningtypeid_13.0', 'airconditioningtypeid_11.0', 'heatingorsystemtypeid_18.0', 'airconditioningtypeid_Unknown', 'buildingqualitytypeid', 'garagecarcnt', 'garagetotalsqft', 'propertylandusetypeid', 'heatingorsystemtypeid_7.0', 'bedroomcnt', 'finishedfloor1squarefeet', 'numberofstories', 'airconditioningtypeid_1.0']
Best Ridge MAE: 232093.05612228555


In [ ]:

#Final evaluation of Ridge using selected features
X_fs_scaled = X_scaled[:, [X.columns.get_loc(f) for f in selected_features]]

ridge_final = Ridge()
cv_mae_scores = -cross_val_score(
    ridge_final,
    X_fs_scaled,
    y,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

print("Ridge (Forward Selection) Mean CV MAE:", np.mean(cv_mae_scores))
print("Ridge (Forward Selection) Std Dev CV MAE:", np.std(cv_mae_scores))

Ridge (Forward Selection) Mean CV MAE: 232093.05612228555
Ridge (Forward Selection) Std Dev CV MAE: 2916.9930813041005


In [17]:
#Model 2: Linear Regression with forward selection

# Use engineered dataset (FE Part 2: log-transform)
X = X_train_fe2.copy()
y = y_train.copy()

# 2. Forward Selection Setup
selected_features = []
remaining_features = list(X.columns)
best_mae = np.inf

cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

# Forward Selection Loop
while remaining_features:
    mae_candidates = []

    for feature in remaining_features:
        trial_features = selected_features + [feature]
        X_trial = X[trial_features]

        linreg = LinearRegression()
        scores = -cross_val_score(
            linreg,
            X_trial,
            y,
            cv=cv,
            scoring='neg_mean_absolute_error'
        )
        mae_candidates.append((feature, scores.mean()))

    # Pick the feature that gives the lowest MAE
    best_feature, best_feature_mae = min(mae_candidates, key=lambda x: x[1])

    # Only add the feature if it improves MAE
    if best_feature_mae < best_mae:
        selected_features.append(best_feature)
        remaining_features.remove(best_feature)
        best_mae = best_feature_mae
    else:
        break

print("Selected features:", selected_features)
print("Best Linear Regression MAE during selection:", best_mae)

# Final Evaluation using Selected Features
X_fs = X[selected_features]

linreg_final = LinearRegression()
cv_mae_scores = -cross_val_score(
    linreg_final,
    X_fs,
    y,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

print("Linear Regression (Forward Selection) Mean CV MAE:", np.mean(cv_mae_scores))
print("Linear Regression (Forward Selection) Std Dev CV MAE:", np.std(cv_mae_scores))


Selected features: ['calculatedfinishedsquarefeet', 'calculatedfinishedsquarefeet_log', 'latitude', 'regionidneighborhood', 'finishedsquarefeet12', 'bathroomcnt', 'propertycountylandusecode', 'heatingorsystemtypeid_24.0', 'lotsizesquarefeet_log', 'airconditioningtypeid_13.0', 'airconditioningtypeid_11.0', 'heatingorsystemtypeid_18.0', 'airconditioningtypeid_Unknown', 'buildingqualitytypeid', 'garagecarcnt', 'garagetotalsqft', 'propertylandusetypeid', 'heatingorsystemtypeid_7.0', 'bedroomcnt', 'finishedfloor1squarefeet', 'numberofstories', 'airconditioningtypeid_1.0']
Best Linear Regression MAE during selection: 232094.8800325672
Linear Regression (Forward Selection) Mean CV MAE: 232094.8800325672
Linear Regression (Forward Selection) Std Dev CV MAE: 2917.2214923334113


In [ ]:
from sklearn.inspection import permutation_importance

#Model 2: HistGradientBoosting with permutation_importance
# Use log-transformed dataset
X = X_train_fe2.copy()
y = y_train.copy()

# Train full HGB model
hgb_full = HistGradientBoostingRegressor(random_state=42)
hgb_full.fit(X, y)

# Compute permutation importance
perm = permutation_importance(
    hgb_full,
    X,
    y,
    n_repeats=5,
    random_state=42
)

importances = pd.Series(perm.importances_mean, index=X.columns)

# Select top 20 features
top_features = importances.sort_values(ascending=False).head(20).index.tolist()
print("Selected HGB features:", top_features)

# Evaluate HGB using selected features
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)
hgb = HistGradientBoostingRegressor(random_state=42)

scores = -cross_val_score(
    hgb,
    X[top_features],
    y,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

print("HGB Mean CV MAE:", scores.mean())
print("HGB Std Dev CV MAE:", scores.std())


Selected HGB features: ['latitude', 'longitude', 'finishedsquarefeet12', 'calculatedfinishedsquarefeet', 'yearbuilt', 'lotsizesquarefeet', 'bathroomcnt', 'buildingqualitytypeid', 'rawcensustractandblock', 'bedroomcnt', 'propertycountylandusecode', 'regionidneighborhood', 'regionidcity', 'regionidzip', 'propertyzoningdesc', 'propertylandusetypeid', 'censustractandblock', 'fullbathcnt', 'airconditioningtypeid_1.0', 'garagetotalsqft']
HGB Mean CV MAE: 192544.67037713586
HGB Std Dev CV MAE: 2942.1117140725078


### 3.B Discussion

#### 3.B.1

Did feature selection improve the validation performance of any of your models?

>Feature selection improved the validation performance of two of the models: **Linear Regression** and **Ridge Regression**.

> Feature selection had different effects depending on the model. It(Forward Selection) improved **Ridge Regression**, reducing the cross-validated MAE from approximately 234,771 to 232,093 while also lowering the standard deviation 3,474 to 2,917, indicating more stable performance. This is expected because Ridge Regression benefits from removing irrelevant or noisy features.

> **Linear Regression** improved significantly after forward feature selection, reducing the MAE from 246,801 to 232,095, an improvement of approximately 14,706 MAE. The model also became more stable, with a lower standard deviation of 2,917. This improvement occurred because forward selection removed noisy, redundant, and highly correlated features that negatively affected Linear Regression. By using a smaller set of more relevant predictors, the model became less sensitive to multicollinearity, more stable, and better able to capture the linear patterns in the data. This was the largest improvement among the models tested.

> For **HistGradientBoosting**, feature selection using permutation importance produced only a minor improvement, reducing the MAE from approximately 192,790 to 192,545, with a slight improvement 2,968 to 2,942 in stability. The gain was minimal because the model was already performing well after the log transformation.


#### 3.B.2

Were there features that were consistently retained (or consistently removed) across multiple models?

> The following features were consistently identified as uninformative across all three models(Ridge, Linear Regression, and HistGradientBoosting) and removed from the final feature set:

>finishedsquarefeet50
>fireplacecnt
>poolsizesum
>roomcnt
>threequarterbathnbr
>unitcnt
>yardbuildingsqft17
>fireplaceflag
>taxdelinquencyflag
>airconditioningtypeid_5.0
>airconditioningtypeid_9.0
>heatingorsystemtypeid_10.0
>heatingorsystemtypeid_11.0
>heatingorsystemtypeid_13.0
>heatingorsystemtypeid_2.0
>heatingorsystemtypeid_6.0
>heatingorsystemtypeid_Unknown

>Interpretation: Features were consistently removed because they provided limited predictive value or introduced noise into the models. Many variables, such as poolsizesum, fireplacecnt, threequarterbathnbr, and yardbuildingsqft17, had very low variance with many zero values, making it difficult for models to learn meaningful patterns. Other features, including roomcnt, unitcnt, and finishedsquarefeet50, contained redundant information already captured by stronger variables such as square footage, bedrooms, and bathrooms. Rare HVAC/heating categories and binary flags like fireplaceflag and taxdelinquencyflag also contributed little predictive information. Overall, forward selection removed sparse, redundant, and weak predictors to improve model efficiency and stability.Since these features were not selected by any of the three models, they appear to contribute little or no predictive value for estimating home prices. Their lack of importance likely reflects weak relationships with the target variable, high sparsity, or noisy information. Removing them simplifies the model, reduces dimensionality, and improves efficiency without significantly affecting predictive performance.

>The features that consistently appeared across Ridge, Linear Regression, and HistGradientBoosting are the strongest predictors of home value. These include:

> Square footage: finishedsquarefeet12, calculatedfinishedsquarefeet
> Bathrooms: bathroomcnt
> Bedrooms: bedroomcnt
> Location: latitude, regionidneighborhood
> Land use: propertycountylandusecode, propertylandusetypeid
> Garage size: garagetotalsqft
>Building quality: buildingqualitytypeid

>These variables consistently demonstrated high predictive power because they directly capture the key factors that influence a property's market value. Larger homes with more living space, bedrooms, and bathrooms generally command higher prices, while neighborhood and geographic location reflect differences in demand, accessibility, school districts, and local amenities. Building quality indicates the condition and construction standard of the property, and garage size adds functional and resale value. Land use codes distinguish different property types, which have different pricing patterns. Since these features were selected by multiple modeling approaches—including both linear and tree-based models—they exhibit robust, model-independent predictive strength and should be retained in the final modeling pipeline.

#### 3.B.3

Were any of your engineered features selected as important? If so, what does this suggest about the hypotheses you developed in Milestone 1?

> Both **Ridge** Regression and **Linear** Regression selected several engineered features, indicating that feature engineering improved the model's ability to capture linear relationships with property values. Both log-transformed variables **calculatedfinishedsquarefeet_log** and **lotsizesquarefeet_log** were identified as important, suggesting that applying logarithmic transformations reduced skewness and created more linear relationships with the target variable. Ridge and Linear Regression also selected multiple one-hot encoded categorical features related to air conditioning and heating system types, demonstrating that converting categorical variables into numerical indicators provided additional predictive information. Feature engineering improved each model's ability to capture linear relationships with property values by reducing skewness, strengthening linearity, and converting categorical information into usable numerical form. Overall, these results show that Ridge and Linear Regression benefit from engineered features because they enhance the representation of the data and improve the model's ability to learn meaningful linear patterns.

>The results provide partial support for the Milestone 1 hypothesis, demonstrating that the effectiveness of feature engineering depends on the type of machine learning model. The hypothesis was strongly validated for Ridge Regression, where the log-transformed features were consistently selected, leading to improved performance and model stability. This confirms that reducing skewness and creating more linear relationships benefits linear models, which rely on these assumptions.

>However, the hypothesis was not supported for **HistGradientBoosting**, as the model selected the log-transformed features and instead favored the original continuous variables. These tree-based models naturally handle skewed data, outliers, and non-linear relationships through threshold-based splits, making log transformations largely unnecessary. Overall, the findings indicate that log transformations are model-specific rather than universally beneficial, providing clear advantages for linear models like Ridge/Linear Regression but offering little or no benefit for tree-based ensemble methods.

#### 3.B.4

After feature selection, did simpler models perform as well as—or better than—the models using the full feature set? Briefly discuss any tradeoffs you observed between model complexity and predictive performance.


> Feature selection and preprocessing had different effects depending on the model. The Ridge Regression model is benefited with its cross-validation MAE improving from 241,750 to 232,093 and its standard deviation decreasing from 3,295 to 2,917. This indicates that removing less informative features and applying log transformations reduced noise, strengthened linear relationships, and improved the model's ability to generalize. Since Ridge is a linear model, it performs better with a smaller set of relevant predictors and less multicollinearity.

> Linear Regression benefited from feature selection and engineered features in a very similar way to Ridge Regression, with its cross-validation MAE improving from 246,801 to 232,095 and its standard deviation decreasing from 3,408 to 2,917, showing that removing noisy predictors and incorporating log-transformed variables strengthened the linear relationships the model relies on. Like Ridge, Linear Regression is highly sensitive to skewness, multicollinearity, and irrelevant features, so applying log transformations to variables reduced distortion in the data and made the underlying patterns more linear, while forward selection ensured that only the most informative predictors were retained.

>These improvements confirm that Linear Regression, as a purely linear model without regularization, benefits even more strongly from engineered features that reduce noise, improve linearity, and simplify the feature space, supporting the hypothesis that feature engineering is especially effective for linear models. Ridge includes L2 regularization, which makes it less sensitive to noise and multicollinearity than Linear Regression. Because of this, Linear Regression benefits even more strongly from feature engineering.

> The HistGradientBoostingRegressor showed only a very small improvement, with MAE decreasing from 192,790 to 192,545 and a slight reduction in standard deviation. Histogram-based gradient boosting already manages skewed data, nonlinear relationships, and feature interactions effectively, so additional feature selection had only a minimal effect on its performance.

> Overall, these results highlight that the effectiveness of feature selection depends on the model. Simpler linear models, such as Ridge and Linear Regression, can benefit significantly from reducing the feature set and improving feature distributions, while more complex tree-based model HistGradientBoostingRegressor generally require less manual feature engineering because they can automatically identify useful splits and interactions. In this project, feature selection and preprocessing clearly improved the Ridge model but provided little to no benefit for the Random Forest and HistGradientBoostingRegressor models.

## Part 4: Tune Your Models [8 pts]

### 4.A Coding

Using the three models developed in **Part 3** (including your final preprocessing, feature engineering, and feature selection decisions):

Investigate whether **hyperparameter tuning** can further improve model performance.

For each of your three models:

* Select one or more important hyperparameters to tune.
* Use one or more appropriate tuning methods. Consider first using validation curves (`sweep_parameter`) to identify a promising region or performance plateau, followed by a focused search using methods such as:

    * GridSearchCV
    * RandomizedSearchCV
    * Another reasonable hyperparameter search method

* Choose hyperparameter values based on the validation results. If several nearby values produce similar validation performance (a performance plateau), prefer **values near the beginning of the plateau,** since they often produce simpler models with nearly identical predictive performance.
* Retrain the model using those hyperparameters.
* Evaluate the tuned model using repeated 5-fold cross-validation (5 repeats).
* Report the validation performance (**mean** and **standard deviation** of the CV MAE).


In [ ]:

# Model 1: Ridge Regression with Forward Selection + Tuning
from sklearn.model_selection import (
    RepeatedKFold, cross_val_score, validation_curve, GridSearchCV
)

# 1. Use log-transformed + engineered dataset
X = X_train_fe2.copy()
y = y_train.copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. Forward Selection
selected_features = []
remaining_features = list(X.columns)
best_mae = np.inf

cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

while remaining_features:
    mae_candidates = []

    for feature in remaining_features:
        trial_features = selected_features + [feature]
        X_trial = X_scaled[:, [X.columns.get_loc(f) for f in trial_features]]

        ridge = Ridge()
        scores = -cross_val_score(
            ridge, X_trial, y, cv=cv, scoring='neg_mean_absolute_error'
        )
        mae_candidates.append((feature, scores.mean()))

    best_feature, best_feature_mae = min(mae_candidates, key=lambda x: x[1])

    if best_feature_mae < best_mae:
        selected_features.append(best_feature)
        remaining_features.remove(best_feature)
        best_mae = best_feature_mae
    else:
        break

print("Selected features:", selected_features)
print("Best Ridge MAE (before tuning):", best_mae)

# 3. SWEEP PARAMETER (Validation Curve)
X_fs_scaled = X_scaled[:, [X.columns.get_loc(f) for f in selected_features]]

alpha_range = np.logspace(-4, 4, 30)  # wide sweep

train_scores, val_scores = validation_curve(
    Ridge(),
    X_fs_scaled,
    y,
    param_name="alpha",
    param_range=alpha_range,
    cv=cv,
    scoring="neg_mean_absolute_error"
)

mean_val_mae = -val_scores.mean(axis=1)

best_alpha_index = np.argmin(mean_val_mae)
best_alpha_from_curve = alpha_range[best_alpha_index]

print("Best alpha from sweep_parameter (validation curve):", best_alpha_from_curve)


# 4. Focused GridSearchCV around plateau
alpha_grid = [
    best_alpha_from_curve / 3,
    best_alpha_from_curve / 2,
    best_alpha_from_curve,
    best_alpha_from_curve * 2,
    best_alpha_from_curve * 3
]

param_grid = {"alpha": alpha_grid}

ridge = Ridge()

grid_search = GridSearchCV(
    ridge,
    param_grid,
    cv=cv,
    scoring="neg_mean_absolute_error"
)

grid_search.fit(X_fs_scaled, y)

best_alpha_final = grid_search.best_params_["alpha"]
print("Final tuned alpha:", best_alpha_final)

# 5. Final evaluation using tuned hyperparameters
ridge_final = Ridge(alpha=best_alpha_final)

cv_mae_scores = -cross_val_score(
    ridge_final,
    X_fs_scaled,
    y,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

print("Ridge (Tuned + FS) Mean CV MAE:", np.mean(cv_mae_scores))
print("Ridge (Tuned + FS) Std Dev CV MAE:", np.std(cv_mae_scores))


Selected features: ['calculatedfinishedsquarefeet', 'calculatedfinishedsquarefeet_log', 'latitude', 'regionidneighborhood', 'finishedsquarefeet12', 'bathroomcnt', 'propertycountylandusecode', 'heatingorsystemtypeid_24.0', 'lotsizesquarefeet_log', 'airconditioningtypeid_13.0', 'airconditioningtypeid_11.0', 'heatingorsystemtypeid_18.0', 'airconditioningtypeid_Unknown', 'buildingqualitytypeid', 'garagecarcnt', 'garagetotalsqft', 'propertylandusetypeid', 'heatingorsystemtypeid_7.0', 'bedroomcnt', 'finishedfloor1squarefeet', 'numberofstories', 'airconditioningtypeid_1.0']
Best Ridge MAE (before tuning): 232093.05612228555
Best alpha from sweep_parameter (validation curve): 2807.2162039411755
Final tuned alpha: 2807.2162039411755
Ridge (Tuned + FS) Mean CV MAE: 230831.52328677822
Ridge (Tuned + FS) Std Dev CV MAE: 2739.8720072103774


In [19]:
# Model 2: Linear Regression with Forward Selection + Tuning
from sklearn.metrics import mean_absolute_error

# Use  final preprocessed + engineered dataset
X = X_train_fe2.copy()
y = y_train.copy()


# Forward Selection
selected_features = []
remaining_features = list(X.columns)
best_mae = np.inf

cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

while remaining_features:
    mae_candidates = []

    for feature in remaining_features:
        trial_features = selected_features + [feature]
        X_trial = X[trial_features].values

        lr = LinearRegression()
        scores = -cross_val_score(
            lr, X_trial, y, cv=cv, scoring='neg_mean_absolute_error'
        )
        mae_candidates.append((feature, scores.mean()))

    best_feature, best_feature_mae = min(mae_candidates, key=lambda x: x[1])

    if best_feature_mae < best_mae:
        selected_features.append(best_feature)
        remaining_features.remove(best_feature)
        best_mae = best_feature_mae
    else:
        break

print("Selected features (initial FS):", selected_features)
print("Best Linear Regression MAE (before tuning):", best_mae)


#  sweep_parameter: tune number of selected features (k)
feature_counts = np.arange(1, len(selected_features) + 1)
val_scores = []

for k in feature_counts:
    X_k = X[selected_features[:k]].values
    lr = LinearRegression()
    scores = cross_val_score(
        lr, X_k, y, cv=cv, scoring="neg_mean_absolute_error"
    )
    val_scores.append(scores)

mean_val_mae = -np.array(val_scores).mean(axis=1)

best_k_index = np.argmin(mean_val_mae)
best_k_from_curve = feature_counts[best_k_index]

print("Best number of features from sweep_parameter:", best_k_from_curve)


# Proper scikit-learn compatible wrapper for GridSearchCV

class LRFeatureSelector:
    """A scikit-learn compatible estimator that selects first k features."""

    def __init__(self, k=1):
        self.k = k
        self.model = LinearRegression()

    def fit(self, X, y):
        self.model.fit(X[:, :self.k], y)
        return self

    def predict(self, X):
        return self.model.predict(X[:, :self.k])

    def get_params(self, deep=True):
        return {"k": self.k}

    def set_params(self, **params):
        self.k = params.get("k", self.k)
        return self

# Custom scorer
def mae_scorer(estimator, X, y):
    return -mean_absolute_error(y, estimator.predict(X))

# Focused GridSearchCV around plateau start

k_grid = [
    max(1, best_k_from_curve - 2),
    max(1, best_k_from_curve - 1),
    best_k_from_curve,
    min(len(selected_features), best_k_from_curve + 1),
    min(len(selected_features), best_k_from_curve + 2)
]

param_grid = {"k": k_grid}

grid_search = GridSearchCV(
    estimator=LRFeatureSelector(),
    param_grid=param_grid,
    cv=cv,
    scoring=mae_scorer
)

grid_search.fit(X[selected_features].values, y)

best_k_final = grid_search.best_params_["k"]
print("Final tuned number of features:", best_k_final)

# Final evaluation using tuned hyperparameters
lr_final = LinearRegression()
X_final = X[selected_features[:best_k_final]].values

cv_mae_scores = -cross_val_score(
    lr_final,
    X_final,
    y,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

print("Linear Regression (Tuned + FS) Mean CV MAE:", np.mean(cv_mae_scores))
print("Linear Regression (Tuned + FS) Std Dev CV MAE:", np.std(cv_mae_scores))

Selected features (initial FS): ['calculatedfinishedsquarefeet', 'calculatedfinishedsquarefeet_log', 'latitude', 'regionidneighborhood', 'finishedsquarefeet12', 'bathroomcnt', 'propertycountylandusecode', 'heatingorsystemtypeid_24.0', 'lotsizesquarefeet_log', 'airconditioningtypeid_13.0', 'airconditioningtypeid_11.0', 'heatingorsystemtypeid_18.0', 'airconditioningtypeid_Unknown', 'buildingqualitytypeid', 'garagecarcnt', 'garagetotalsqft', 'propertylandusetypeid', 'heatingorsystemtypeid_7.0', 'bedroomcnt', 'finishedfloor1squarefeet', 'numberofstories', 'airconditioningtypeid_1.0']
Best Linear Regression MAE (before tuning): 232094.88003256373
Best number of features from sweep_parameter: 22
Final tuned number of features: 22
Linear Regression (Tuned + FS) Mean CV MAE: 232094.88003256373
Linear Regression (Tuned + FS) Std Dev CV MAE: 2917.221492439287


In [10]:
# Model 3: HistGradientBoostingRegressor + Feature Selection + Tuning
from sklearn.model_selection import validation_curve
from sklearn.inspection import permutation_importance
# 1. Use engineered dataset
X = X_train_fe2.copy()
y = y_train.copy()

# 2. Feature Selection using Permutation Importance
hgb_full = HistGradientBoostingRegressor(random_state=42)
hgb_full.fit(X, y)

perm = permutation_importance(
    hgb_full,
    X,
    y,
    n_repeats=5,
    random_state=42
)

importances = pd.Series(perm.importances_mean, index=X.columns)

top_features = importances.sort_values(ascending=False).head(20).index.tolist()
print("Selected HGB features:", top_features)

X_fs = X[top_features]

# 3. SWEEP PARAMETER (Validation Curve)
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

# Sweep learning_rate
lr_range = np.logspace(-3, 0, 15)

train_scores, val_scores = validation_curve(
    HistGradientBoostingRegressor(random_state=42),
    X_fs,
    y,
    param_name="learning_rate",
    param_range=lr_range,
    cv=cv,
    scoring="neg_mean_absolute_error"
)

mean_val_mae = -val_scores.mean(axis=1)

best_index = np.argmin(mean_val_mae)
best_lr_from_curve = lr_range[best_index]

print("Best learning_rate from sweep_parameter:", best_lr_from_curve)

# 4. Focused Hyperparameter Tuning (RandomizedSearchCV)
param_dist = {
    "learning_rate": [
        best_lr_from_curve / 3,
        best_lr_from_curve / 2,
        best_lr_from_curve,
        best_lr_from_curve * 2
    ],
    "max_depth": [3, 5, 7, None],
    "max_leaf_nodes": [15, 31, 63],
    "min_samples_leaf": [1, 5, 10]
}

hgb = HistGradientBoostingRegressor(random_state=42)

random_search = RandomizedSearchCV(
    hgb,
    param_distributions=param_dist,
    n_iter=20,
    cv=cv,
    scoring="neg_mean_absolute_error",
    random_state=42
)

random_search.fit(X_fs, y)

best_params = random_search.best_params_
print("Tuned HGB hyperparameters:", best_params)

# 5. Final Evaluation using Tuned Hyperparameters
hgb_final = HistGradientBoostingRegressor(**best_params, random_state=42)

cv_mae_scores = -cross_val_score(
    hgb_final,
    X_fs,
    y,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

print("HGB (Tuned + FS) Mean CV MAE:", np.mean(cv_mae_scores))
print("HGB (Tuned + FS) Std Dev CV MAE:", np.std(cv_mae_scores))


Selected HGB features: ['latitude', 'longitude', 'finishedsquarefeet12', 'calculatedfinishedsquarefeet', 'yearbuilt', 'lotsizesquarefeet', 'bathroomcnt', 'buildingqualitytypeid', 'rawcensustractandblock', 'bedroomcnt', 'propertycountylandusecode', 'regionidneighborhood', 'regionidcity', 'regionidzip', 'propertyzoningdesc', 'propertylandusetypeid', 'censustractandblock', 'fullbathcnt', 'airconditioningtypeid_1.0', 'garagetotalsqft']
Best learning_rate from sweep_parameter: 0.13894954943731375
Tuned HGB hyperparameters: {'min_samples_leaf': 5, 'max_leaf_nodes': 63, 'max_depth': 7, 'learning_rate': np.float64(0.13894954943731375)}
HGB (Tuned + FS) Mean CV MAE: 190881.44463701118
HGB (Tuned + FS) Std Dev CV MAE: 2961.474381805629


### 4.B Discussion

Answer the following questions.

#### 4.B.1

Which hyperparameters had the greatest impact on model performance? Briefly explain.

>The hyperparameters that had the greatest impact on model performance were those that directly control model flexibility and regularization, and their effects differed sharply across Ridge, Linear Regression, and HistGradientBoostingRegressor. For Ridge Regression, the most influential hyperparameter was alpha, which determines the strength of L2 regularization; tuning alpha to 2807.21 reduced the MAE from 232,093 to 230,832 and lowered the standard deviation, showing that stronger regularization helped stabilize coefficients and reduce overfitting. For Linear Regression, the key hyperparameter was the number of selected features, with the optimal value being 22; unlike Ridge, Linear Regression has no regularization, so controlling model complexity through feature count was the only meaningful tuning mechanism, and performance remained unchanged after tuning because forward selection had already identified the optimal subset. For HistGradientBoostingRegressor, the hyperparameters with the greatest impact were learning_rate, max_depth, and max_leaf_nodes, which govern how aggressively the model fits nonlinear patterns; tuning these parameters—especially lowering the learning rate to 0.1389—produced the largest improvement of all models, reducing MAE from 192,545 to 190,881. Overall, the most impactful hyperparameters were those that controlled regularization for Ridge, feature count for Linear Regression, and tree complexity and learning rate for HistGradientBoostingRegressor.

#### 4.B.2

Did hyperparameter tuning substantially improve the performance of all three models, or only some of them?

> Hyperparameter tuning did not substantially improve all three models — it helped some models a lot, helped one model moderately, and had almost no effect on another. The impact depended entirely on how much flexibility each model already had and how sensitive it was to its tuning parameters. Ridge Regression showed a meaningful improvement, with MAE dropping from 232,093 to 230,832, because tuning alpha directly controls the strength of regularization and helped the model balance bias and variance more effectively. Linear Regression, however, saw no improvement at all from tuning, because it has no regularization hyperparameters and its performance is determined entirely by the selected feature subset; once forward selection chose the best 22 features, there was nothing left to tune. In contrast, HistGradientBoostingRegressor experienced the largest performance gain, improving from 192,545 to 190,881 MAE, because tuning hyperparameters like learning_rate, max_depth, and max_leaf_nodes directly affects how deeply and aggressively the model learns nonlinear patterns. Overall, hyperparameter tuning mattered a lot for HGB, moderately for Ridge, and not at all for Linear Regression, highlighting that tuning is most impactful for flexible, nonlinear models and least impactful for simple linear ones.

#### 4.B.3

Which tuning method(s) did you use for each model? Briefly explain why you chose those methods.

> **Ridge Regression — Validation Curve for Alpha**
For Ridge Regression, I have used a validation curve to tune the regularization strength (alpha). A validation curve systematically evaluates model performance across a range of alpha values, allowing us to see how increasing or decreasing regularization affects bias, variance, and overall MAE. This method is ideal for Ridge because it has one dominant hyperparameter, and the relationship between alpha and model performance is often smooth and predictable. By sweeping through many alpha values, you identified the point where the model achieved the best balance between underfitting and overfitting. After locating the optimal region, I applied a focused GridSearchCV around the plateau to refine the final alpha value, ensuring that the model was tuned precisely without unnecessary computation. This combination provided clear insight into how regularization stabilizes the model and made it the most appropriate tuning strategy for Ridge.

>**Linear Regression — Feature Count Sweep During Forward Selection**
Linear Regression has no regularization hyperparameters, so traditional tuning methods like grid search or validation curves are not meaningful. Instead, the only way to control model complexity is by adjusting the number of selected features during forward selection. For Linear Regression, I have tuned the model by sweeping over different numbers of selected features during forward selection. Because Linear Regression has no regularization hyperparameters, the only meaningful way to control its complexity is by adjusting how many predictors the model uses. The feature-count sweep allowed us to evaluate MAE across subsets of increasing size, helping identify the point where adding more features stopped improving performance. After locating the optimal region, I applied a focused GridSearchCV around the plateau to refine the final number of features, ensuring that the model used the smallest and most informative subset. This combination was the most appropriate tuning strategy for Linear Regression because it directly targeted the model's only source of complexity, its feature set, and prevented overfitting while preserving interpretability.

> **HistGradientBoostingRegressor + Feature Selection + Tuning**
For HistGradientBoostingRegressor, I used a two stage tuning strategy that reflects the model's complexity and the interaction between its hyperparameters. First, I applied a validation curve to sweep over a wide range of learning_rate values. This was essential because learning_rate is the most influential hyperparameter in gradient boosting: it controls how aggressively each boosting iteration fits residual errors. By evaluating MAE across a logarithmic range of learning rates, I identified the region where the model achieved the best balance between underfitting (learning_rate too small) and overfitting (learning_rate too large). Once the optimal learning_rate region was found, I used RandomizedSearchCV to tune the remaining hyperparameters—max_depth, max_leaf_nodes, and min_samples_leaf—alongside refined learning_rate candidates. RandomizedSearchCV was the appropriate choice because HGB has multiple interacting hyperparameters, and random sampling allows efficient exploration of a large search space without the computational cost of exhaustive grid search. This combination of a targeted validation curve followed by a broad randomized search enabled the model to capture nonlinear patterns more effectively and produced the largest performance improvement of all three models.

#### 4.B.4

After tuning, how did the relative performance of your three models change? Did tuning affect which model appeared to perform best?

> After tuning, the performance gap between the three models became clearer and more pronounced, and tuning reinforced not changed—which model was the strongest. Ridge Regression improved modestly, Linear Regression stayed essentially the same, and HistGradientBoostingRegressor improved substantially, widening its lead over the linear models. **Ridge's** MAE dropped from 232,093 to 230,832, and its standard deviation decreased, showing that tuning alpha helped the model regularize more effectively and generalize slightly better. **Linear Regression**, however, showed no meaningful improvement after tuning because it has no regularization hyperparameters; its performance is determined entirely by the selected feature subset, and the feature-count sweep simply confirmed that the forward-selected 22 features were already optimal. In contrast, **HistGradientBoostingRegressor** experienced the largest gain, improving from 192,545 to 190,881 MAE after tuning learning_rate, max_depth, max_leaf_nodes, and min_samples_leaf. These hyperparameters directly control how deeply and aggressively the model learns nonlinear patterns, so tuning them allowed HGB to extract more structure from the data without overfitting. Overall, tuning did not change which model performed best, HGB was already the strongest but it increased the performance gap, making the nonlinear model even more dominant relative to Ridge and Linear Regression.

## Part 5: Final Model and Workflow Assessment [14 pts]

### 5.A Coding

Using the work completed in **Parts 1–4**:

Select your **best-performing model** and prepare your final modeling pipeline.

Your pipeline should include all preprocessing, feature engineering, feature selection, and hyperparameter tuning decisions that you chose to retain.

Evaluate your final model by:

* Training on the complete training dataset.
* Reporting the **mean** and **standard deviation** of the repeated cross-validation MAE.
* Evaluating the model on the held-out test set.
* Reporting the final test MAE.

In [3]:
# Preprocessing:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
#Final lost of columns to be dropped based on above three steps:
cols_to_drop= [
    'buildingclasstypeid',
    'finishedsquarefeet13',
    'basementsqft',
    'storytypeid',
    'yardbuildingsqft26',
    #'fireplaceflag',
    'architecturalstyletypeid',
    'typeconstructiontypeid',
    'finishedsquarefeet6',
    'pooltypeid10',
    'decktypeid',
    #'poolsizesum',
    'pooltypeid2',
    'hashottuborspa',
    'taxdelinquencyyear',
    #'taxdelinquencyflag',
    'finishedsquarefeet15',
'parcelid', 'fips', 'assessmentyear', 'regionidcounty',#'rawcensustractandblock', 'censustractandblock',
                    #'hashottuborspa'
                    'pooltypeid7','poolcnt']

df_reduced = df.drop(columns=cols_to_drop)
# Remove Problematic Samples
df_clean = df_reduced.copy()

# 1. Remove data samples with missing target values
df_clean = df_clean[df_clean["taxvaluedollarcnt"].notna()]

# 2. Remove data samples with >90% missing features
row_missing_pct = df_clean.isna().mean(axis=1) * 100
df_clean = df_clean[row_missing_pct <= 90]

print("Remaining rows after cleaning:", df_clean.shape[0])
# Verify the new shape
print("Original shape:", df.shape)
print("New shape:", df_clean.shape)

# Split the Dataset into Training and Test Sets
X = df_clean.drop(columns=["taxvaluedollarcnt"])
y = df_clean["taxvaluedollarcnt"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# 1. Identify column types
ordinal_cols = ['buildingqualitytypeid']

categorical_cols = [
    'airconditioningtypeid',
    'heatingorsystemtypeid',
    'propertycountylandusecode',
    'propertylandusetypeid',
    'propertyzoningdesc',
    'regionidcity',
    'regionidneighborhood',
    'regionidzip',
    'fireplaceflag',
    'taxdelinquencyflag']

numeric_cols = [col for col in X_train.columns
                if col not in categorical_cols + ordinal_cols]

# 2. Fit imputers on TRAIN only

# Numeric imputer (median)
num_imputer = SimpleImputer(strategy="median")
num_imputer.fit(X_train[numeric_cols])

# Ordinal imputer (median)
ord_imputer = SimpleImputer(strategy="median")
ord_imputer.fit(X_train[ordinal_cols])

# Categorical imputer ("Unknown")
cat_imputer = SimpleImputer(strategy="constant", fill_value="Unknown")
cat_imputer.fit(X_train[categorical_cols])

# 3. Transform TRAIN + TEST
# Numeric
X_train_num = pd.DataFrame(
    num_imputer.transform(X_train[numeric_cols]),
    columns=numeric_cols,
    index=X_train.index)

X_test_num = pd.DataFrame(
    num_imputer.transform(X_test[numeric_cols]),
    columns=numeric_cols,
    index=X_test.index)

# Ordinal
X_train_ord = pd.DataFrame(
    ord_imputer.transform(X_train[ordinal_cols]),
    columns=ordinal_cols,
    index=X_train.index)

X_test_ord = pd.DataFrame(
    ord_imputer.transform(X_test[ordinal_cols]),
    columns=ordinal_cols,
    index=X_test.index)

# Categorical
X_train_cat = pd.DataFrame(
    cat_imputer.transform(X_train[categorical_cols]),
    columns=categorical_cols,
    index=X_train.index)

X_test_cat = pd.DataFrame(
    cat_imputer.transform(X_test[categorical_cols]),
    columns=categorical_cols,
    index=X_test.index)

# 4. Final combined datasets
X_train_imputed = pd.concat([X_train_num, X_train_ord, X_train_cat], axis=1)
X_test_imputed = pd.concat([X_test_num, X_test_ord, X_test_cat], axis=1)

print("Missing values in TRAIN:", X_train_imputed.isna().sum().sum())
print("Missing values in TEST:", X_test_imputed.isna().sum().sum())

ordinal_cols = ['buildingqualitytypeid']

high_card_cols = [
    'regionidcity',
    'regionidneighborhood',
    'regionidzip',
    'propertycountylandusecode',
    'propertylandusetypeid',
    'propertyzoningdesc'
]

low_card_cols = [
    'airconditioningtypeid',
    'heatingorsystemtypeid'
]

boolean_cols = [
    'fireplaceflag',
    'taxdelinquencyflag'
]

numeric_cols = [
    col for col in X_train_imputed.columns
    if col not in ordinal_cols + high_card_cols + low_card_cols + boolean_cols
]

convert_to_str = high_card_cols + low_card_cols + boolean_cols

X_train_imputed[convert_to_str] = X_train_imputed[convert_to_str].fillna("Unknown").astype(str)
X_test_imputed[convert_to_str]  = X_test_imputed[convert_to_str].fillna("Unknown").astype(str)

# Ordinal stays numeric
X_train_imputed[ordinal_cols] = X_train_imputed[ordinal_cols].fillna(-1)
X_test_imputed[ordinal_cols]  = X_test_imputed[ordinal_cols].fillna(-1)

ord_enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
ord_enc.fit(X_train_imputed[ordinal_cols])

X_train_ord = pd.DataFrame(ord_enc.transform(X_train_imputed[ordinal_cols]),
                           columns=ordinal_cols, index=X_train_imputed.index)

X_test_ord = pd.DataFrame(ord_enc.transform(X_test_imputed[ordinal_cols]),
                          columns=ordinal_cols, index=X_test_imputed.index)
def frequency_encode(train, test, cols):
    for col in cols:
        freq = train[col].value_counts() / len(train)
        train[col] = train[col].map(freq)
        test[col]  = test[col].map(freq).fillna(0)
    return train, test

X_train_freq = X_train_imputed[high_card_cols].copy()
X_test_freq  = X_test_imputed[high_card_cols].copy()

X_train_freq, X_test_freq = frequency_encode(X_train_freq, X_test_freq, high_card_cols)

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(X_train_imputed[low_card_cols])

X_train_ohe = pd.DataFrame(ohe.transform(X_train_imputed[low_card_cols]),
                           columns=ohe.get_feature_names_out(low_card_cols),
                           index=X_train_imputed.index)

X_test_ohe = pd.DataFrame(ohe.transform(X_test_imputed[low_card_cols]),
                          columns=ohe.get_feature_names_out(low_card_cols),
                          index=X_test_imputed.index)
bool_map = {
    "Y": 1, "N": 0, "Unknown": -1,
    "True": 1, "False": 0,
    True: 1, False: 0
}

X_train_bool = X_train_imputed[boolean_cols].replace(bool_map)
X_test_bool  = X_test_imputed[boolean_cols].replace(bool_map)

X_train_numeric = X_train_imputed[numeric_cols]
X_test_numeric  = X_test_imputed[numeric_cols]


# Replace original flag columns with numeric versions
X_train_imputed[boolean_cols] = X_train_imputed[boolean_cols].replace(bool_map)
X_test_imputed[boolean_cols]  = X_test_imputed[boolean_cols].replace(bool_map)
X_train_final = pd.concat([
    X_train_numeric,
    X_train_ord,
    X_train_freq,
    X_train_ohe,
    X_train_imputed[boolean_cols]   # numeric flags
], axis=1)

X_test_final = pd.concat([
    X_test_numeric,
    X_test_ord,
    X_test_freq,
    X_test_ohe,
    X_test_imputed[boolean_cols]    # numeric flags
], axis=1)

non_numeric = X_train_final.select_dtypes(include=['object'])
print(non_numeric.columns.tolist())



Remaining rows after cleaning: 77578
Original shape: (77613, 55)
New shape: (77578, 35)
Train shape: (62062, 34)
Test shape: (15516, 34)
Missing values in TRAIN: 0
Missing values in TEST: 0


/tmp/ipykernel_1365/988329921.py:157: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_train_imputed[convert_to_str] = X_train_imputed[convert_to_str].fillna("Unknown").astype(str)
/tmp/ipykernel_1365/988329921.py:158: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_test_imputed[convert_to_str]  = X_test_imputed[convert_to_str].fillna("Unknown").astype(str)
/tmp/ipykernel_1365/988329921.py:200: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_o

[]


In [4]:
# Model: HistGradientBoostingRegressor + Feature Selection + Tuning
from sklearn.model_selection import validation_curve
from sklearn.inspection import permutation_importance

#1. Feature Engineering Log transformation using preprocessed data
X_train_fe2 = X_train_final.copy()

skewed_cols = ["lotsizesquarefeet", "calculatedfinishedsquarefeet"]

for col in skewed_cols:
    X_train_fe2[col + "_log"] = np.log1p(X_train_fe2[col])


# Use engineered dataset
X = X_train_fe2.copy()
y = y_train.copy()

# 2. Feature Selection using Permutation Importance
hgb_full = HistGradientBoostingRegressor(random_state=42)
hgb_full.fit(X, y)

perm = permutation_importance(
    hgb_full,
    X,
    y,
    n_repeats=5,
    random_state=42
)

importances = pd.Series(perm.importances_mean, index=X.columns)

top_features = importances.sort_values(ascending=False).head(20).index.tolist()
print("Selected HGB features:", top_features)

Selected HGB features: ['latitude', 'longitude', 'finishedsquarefeet12', 'calculatedfinishedsquarefeet', 'yearbuilt', 'lotsizesquarefeet', 'bathroomcnt', 'buildingqualitytypeid', 'rawcensustractandblock', 'bedroomcnt', 'propertycountylandusecode', 'regionidneighborhood', 'regionidcity', 'regionidzip', 'propertyzoningdesc', 'propertylandusetypeid', 'censustractandblock', 'fullbathcnt', 'airconditioningtypeid_1.0', 'garagetotalsqft']


In [5]:
#Used top 20 features
X_fs = X[top_features]

# SWEEP PARAMETER (Validation Curve)
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

# Sweep learning_rate
lr_range = np.logspace(-3, 0, 15)

train_scores, val_scores = validation_curve(
    HistGradientBoostingRegressor(random_state=42),
    X_fs,
    y,
    param_name="learning_rate",
    param_range=lr_range,
    cv=cv,
    scoring="neg_mean_absolute_error"
)

mean_val_mae = -val_scores.mean(axis=1)
best_index = np.argmin(mean_val_mae)
best_lr_from_curve = lr_range[best_index]

print("Best learning_rate from sweep_parameter:", best_lr_from_curve)

# Focused Hyperparameter Tuning (RandomizedSearchCV)
param_dist = {
    "learning_rate": [
        best_lr_from_curve / 3,
        best_lr_from_curve / 2,
        best_lr_from_curve,
        best_lr_from_curve * 2
    ],
    "max_depth": [3, 5, 7, None],
    "max_leaf_nodes": [15, 31, 63],
    "min_samples_leaf": [1, 5, 10]
}

hgb = HistGradientBoostingRegressor(random_state=42)

random_search = RandomizedSearchCV(
    hgb,
    param_distributions=param_dist,
    n_iter=20,
    cv=cv,
    scoring="neg_mean_absolute_error",
    random_state=42
)

random_search.fit(X_fs, y)

best_params = random_search.best_params_
print("Tuned HGB hyperparameters:", best_params)

#Final Evaluation using Tuned Hyperparameters
hgb_final = HistGradientBoostingRegressor(**best_params, random_state=42)

cv_mae_scores = -cross_val_score(
    hgb_final,
    X_fs,
    y,
    cv=cv,
    scoring='neg_mean_absolute_error'
)

print("HGB (Tuned + FS) Mean CV MAE:", np.mean(cv_mae_scores))
print("HGB (Tuned + FS) Std Dev CV MAE:", np.std(cv_mae_scores))


Best learning_rate from sweep_parameter: 0.13894954943731375
Tuned HGB hyperparameters: {'min_samples_leaf': 5, 'max_leaf_nodes': 63, 'max_depth': 7, 'learning_rate': np.float64(0.13894954943731375)}
HGB (Tuned + FS) Mean CV MAE: 190881.44463701118
HGB (Tuned + FS) Std Dev CV MAE: 2961.474381805629


In [6]:
# Final Test‑Set Evaluation
from sklearn.metrics import mean_absolute_error

#Feature engineering Log-transform skewed numeric features with test data
X_test_fe2 = X_test_final.copy()

skewed_cols = ["lotsizesquarefeet", "calculatedfinishedsquarefeet"]

for col in skewed_cols:
    X_test_fe2[col + "_log"] = np.log1p(X_test_fe2[col])

#Copy the test set
X_test = X_test_fe2.copy()
y_test = y_test.copy()

#Select the same 20 features used in training
top_features = [
    'latitude', 'longitude', 'finishedsquarefeet12', 'calculatedfinishedsquarefeet',
    'yearbuilt', 'lotsizesquarefeet', 'bathroomcnt', 'buildingqualitytypeid',
    'rawcensustractandblock', 'bedroomcnt', 'propertycountylandusecode',
    'regionidneighborhood', 'regionidcity', 'regionidzip', 'propertyzoningdesc',
    'propertylandusetypeid', 'censustractandblock', 'fullbathcnt',
    'airconditioningtypeid_1.0', 'garagetotalsqft'
]

X_test_fs = X_test[top_features]

#Use tuned hyperparameters
best_params = {
    'learning_rate': 0.13894954943731375,
    'max_depth': 7,
    'max_leaf_nodes': 63,
    'min_samples_leaf': 5
}

hgb_final = HistGradientBoostingRegressor(**best_params, random_state=42)

#Fit on full training data (using selected features)
hgb_final.fit(X_fs, y)

#Predict on the held-out test set
y_pred_test = hgb_final.predict(X_test_fs)

#Compute test MAE
test_mae = mean_absolute_error(y_test, y_pred_test)
print("Final HGB Test MAE:", test_mae)


Final HGB Test MAE: 196754.29029291202


### 5.B Discussion

Answer the following questions.

#### 5.B.1

Compare the performance of your final model with its original baseline from **Part 1**. Which changes contributed the most to the improvement?

> The final HistGradientBoostingRegressor(HGB) improved upon the baseline, reducing the mean CV MAE from 192,789.70 to 190,881.44 while slightly lowering the standard deviation from 2,968.20 to 2,961.47, demonstrating both better accuracy and stable generalization. The held-out test MAE of 196,754.29 confirmed that the model generalized well to unseen data. The largest improvement came from permutation-importance feature selection, which retained the top 20 most informative features and removed noisy or redundant predictors. A learning-rate sweep further improved the model by identifying a more effective learning rate (0.1389), leading to a better bias-variance tradeoff. Finally, RandomizedSearchCV refined key hyperparameters (max_depth = 7, max_leaf_nodes = 63, min_samples_leaf = 5), allowing the model to capture more complex nonlinear relationships while maintaining good generalization. Overall, these enhancements widened HGB's performance advantage over the linear models, with feature selection contributing the greatest improvement.

#### 5.B.2

Looking back at the hypotheses you proposed in **Milestone 1**, which were supported by your experimental results? Were any hypotheses disproved?

> Several of the hypotheses proposed in Milestone-1 were only partially supported once evaluated through the performance of the HistGradientBoostingRegressor. The hypothesis that removing highly correlated features would improve model performance was not strongly supported for HGB, because tree-based models naturally handle multicollinearity and do not suffer from unstable coefficients the way linear models do. Likewise, the hypothesis that log-transforming skewed variables would improve model accuracy was not supported: none of the log-transformed features appeared in the top 20 permutation-importance features selected for the final HGB model, and the tuned model performed best using raw continuous variables. Interaction and ratio features also did not meaningfully improve HGB, as the model already captures nonlinear relationships and interactions through its tree-splitting structure. Instead, the improvements in HGB performance came primarily from feature selection—reducing the feature set to the 20 most important predictors—and hyperparameter tuning, especially optimizing the learning rate and tree structure parameters. These steps reduced noise, improved generalization, and produced a substantial MAE improvement over the baseline model. In summary, while many Milestone-1 preprocessing hypotheses were valuable for linear models, the HGB results showed that nonlinear tree-based models benefit far more from targeted feature selection and tuning than from transformations or engineered features.

#### 5.B.3

Why did you select this model as your final model? Discuss both its predictive performance and any other considerations (such as stability, simplicity, or interpretability).

> **HistGradientBoostingRegressor** was selected as the final model because it was the only model that consistently delivered high predictive accuracy, low variance, and strong generalization to the held-out test set even after extensive preprocessing, feature engineering, and tuning of the linear models. It offered the strongest combination of accuracy, stability, and practical interpretability among all the evaluated models. Each of these advantages is meaningful on its own, but together they make HGB the most reliable and effective choice for the tax assessed property valuation prediction.

>**Accuracy**: HGB delivered the lowest prediction error of all models, both before and after tuning. Its final repeated-CV MAE of 190,881 was dramatically better than Ridge (230,832) and Linear Regression (232,093). This improvement comes from HGB's ability to automatically learn nonlinear relationships, threshold effects, and complex interactions among features patterns that linear models cannot capture even with log transforms or engineered interactions. The model also generalized well to unseen data, achieving a held-out test MAE of 196,754, which is close to its cross-validated performance. This consistency shows that HGB is not just accurate on the training folds, it learns meaningful structure in the data that transfers to new properties.

>**Stability**: HGB also demonstrated excellent stability, with a repeated-CV standard deviation of 2,961, slightly lower than the baseline and lower than both linear models. This means its predictions were consistent across different train/test splits, indicating that the model is not overly sensitive to sampling variation. The stability improved further after feature selection, because removing noisy or redundant predictors helped the model focus on the strongest signals. The tuned hyperparameters especially min_samples_leaf=5 and max_leaf_nodes=63 also contributed to stability by preventing overly deep or overly specific trees. Together, these choices produced a model that is both accurate and dependable.

>**Practical Interpretability**: Although HGB is less interpretable than linear models, it still offers practical interpretability through tools like permutation importance, which was used to identify the top 20 most influential features. This allowed to reduce the model to a compact, meaningful subset of predictors such as latitude, longitude, square footage, year built, and neighborhood identifiers that align with domain knowledge and make intuitive sense for property valuation. By focusing on these features, the final model is easier to explain to stakeholders and avoids the complexities often associated with more complex models. The reduced feature set also simplifies deployment and reduces noise, making the model more robust in practice.

>HGB consistently outperformed linear models even after extensive preprocessing and tuning, demonstrating that nonlinear tree-based methods are better suited for capturing the complex relationships present in the Zillow dataset.

#### 5.B.4

What did you learn about your dataset and the machine learning process through this end-to-end modeling workflow? If you had additional time, what would you investigate next?

> I learned a lot about both the dataset and the full machine-learning workflow through this end-to-end project, and the lessons are surprisingly rich once I step back and connect all the pieces. The Zillow dataset taught me that real-world housing data is messy, highly skewed, and full of correlated and redundant variables, which means raw features rarely perform well without thoughtful preprocessing. I observed that geographic identifiers like latitude, longitude, neighborhood, and ZIP code carry far more predictive power than many traditional home-characteristic features, and that engineered features such as log transforms or interactions matter only for certain model families. I have also learned that tree-based models like HistGradientBoostingRegressor naturally capture nonlinear structure and interactions, making many engineered features unnecessary.

>I learned that feature selection turned out to be one of the most powerful steps in the workflow. When I reduced the dataset to the 20 most important predictors, the model became noticeably more accurate and more stable. This happened because tree-based models like feature selection benefit from removing weak, redundant, or noisy features that distract the model from the strongest signals. The top features were mostly geographic and structural variables—latitude, longitude, neighborhood, ZIP code, square footage, year built showing that the dataset's predictive power is concentrated in a relatively small set of meaningful attributes. By focusing on these, the final HGB model learned clearer patterns, avoided overfitting, and generalized better to the test set.

> From a modeling-process perspective, I learned that feature selection can be more valuable than feature creation: removing noise and focusing on the strongest predictors gave the model a clearer signal and improved generalization. I also observed how hyperparameter tuning especially tuning learning rate and tree structure can dramatically improve the performance when done systematically. Repeated cross-validation taught me to evaluate not just accuracy but stability, revealing how sensitive each model was to different train/test splits. Finally, I learned that the best model is not always the most interpretable or the simplest; instead, it's the model that balances predictive accuracy, stability, and practical interpretability, which is why HGB emerged as a final choice.

> Linear models showed higher variance and were more affected by multicollinearity and noisy features. HGB, on the other hand, maintained a low standard deviation across repeated folds, and stability improved even further after feature selection and tuning. This taught me that a good model is not just one with a low MAE, it's one that performs consistently across different samples of the data. Stability is what makes a model trustworthy in real-world deployment.

> Future work on the Zillow housing dataset could focus on further improving prediction accuracy through advanced modeling and feature engineering. Testing more powerful gradient boosting algorithms such as XGBoost, LightGBM, and CatBoost may determine whether the current performance is limited by the model or the data itself. Since geographic variables (e.g., latitude, longitude, neighborhood, and ZIP code) were among the most important predictors, creating additional location-based features—such as neighborhood clusters, distance-to-city-center measures, or interactions between location and property characteristics—could capture more spatial variation in housing prices. Applying a log transformation to the target variable (taxvaluedollarcnt) may also reduce skewness and improve prediction accuracy. Additionally, model stacking could combine the strengths of linear models and HistGradientBoostingRegressor to achieve better overall performance. Finally, conducting error analysis to identify properties that are consistently mispredicted (such as luxury homes or rural properties) and evaluating temporal drift if data from multiple years are available would help improve the model's robustness and generalization for real-world Zillow property valuation.